# Final Project Notebook

DS 5001 Text as Data


# Metadata

- Full Name: Joseph Kaminetz
- Userid: emv3bc
- GitHub Repo URL: https://github.com/Joekam03/TextProject
- UVA Box URL:

# Overview

The goal of the final project is for you to create a **digital analytical edition** of a corpus using the tools, practices, and perspectives you’ve learning in this course. You will select a corpus that has already been digitized and transcribed, parse that into an F-compliant set of tables, and then generate and visualize the results of a series of fitted models. You will also draw some tentative conclusions regarding the linguistic, cultural, psychological, or historical features represented by your corpus. The point of the exercise is to have you work with a corpus through the entire pipeline from ingestion to interpretation. 

Specifically, you will acquire a collection of long-form texts and perform the following operations:

- **Convert** the collection from their source formats (F0) into a set of tables that conform to the Standard Text Analytic Data Model (F2).
- **Annotate** these tables with statistical and linguistic features using NLP libraries such as NLTK (F3).
- **Produce** a vector representation of the corpus to generate TFIDF values to add to the TOKEN (aka CORPUS) and VOCAB tables (F4).
- **Model** the annotated and vectorized model with tables and features derived from the application of unsupervised methods, including PCA, LDA, and word2vec (F5).
- **Explore** your results using statistical and visual methods.
- **Present** conclusions about patterns observed in the corpus by means of these operations.

When you are finished, you will make the results of your work available in GitHub (for code) and UVA Box (for data). You will submit to Gradescope (via Canvas) a PDF version of a Jupyter notebook that contains the information listed below.

# Some Details

- Please fill out your answers in each task below by editing the markdown cell. 
- Replace text that asks you to insert something with the thing, i.e. replace `(INSERT IMAGE HERE)` with an image element, e.g. `![](image.png)`.
- For URLs, just paste the raw URL directly into the text area. Don't worry about providing link labels using `[label](link)`.
- Please do not alter the structure of the document or cell, i.e. the bulleted lists. 
- You may add explanatory paragraphs below the bulleted lists.
- Please name your tables as they are named in each task below.
- Tasks are indicated by headers with point values in parentheses.

# Raw Data

## Source Description (1)

Provide a brief description of your source material, including its provenance and content. Tell us where you found it and what kind of content it contains.

This corpus contains plaintext national constitution documents from the GitHub repository `marcomorucci/Clustering-Constitutions`, specifically the `constitutions` directory. According to the Github repository, the constitutions were originally sourced from https://www.constituteproject.org/. Each source document is a country constitution identified by country name and year, such as `Afghanistan_2004.txt`. The content is legal and governmental prose: preambles, articles, chapters, sections, rights, duties, institutional rules, and amendment or enforcement provisions. 


## Source Features (1)

Add values for the following items. (Do this for all following bulleted lists.)

- Source URL: https://github.com/marcomorucci/Clustering-Constitutions/tree/master/constitutions
- UVA Box URL:
- Number of raw documents: 192
- Total size of raw documents (e.g. in MB): approximately 26.5 MB / 26,515,008 characters
- File format(s), e.g. XML, plaintext, etc.: plaintext `.txt`


## Source Document Structure (1)

Provide a brief description of the internal structure of each document. That, describe the typical elements found in document and their relation to each other. For example, a corpus of literary texts may be organized into chapters, paragraphs, sentences, and tokens.

The documents are parsed with the OHCO structure `doc -> div1 -> div2 -> unit -> para -> sent -> token`. The parser treats titles and parts as high-level `div1` divisions, chapters and sections as `div2` divisions when present, and articles as the main `unit` level. For constitutions without article headings, numbered provisions are used as the unit fallback. Paragraphs are created from non-heading text blocks, then split into sentences and word tokens. A variable OHCO structure like this was chosen because not every constitution has a defined hierarchical structure. Using a variable OHCO allows our analysis to capture whatever structure is present in the database. Codex helped me to automate the parsing of the data using the explained OHCO structure.


# Parsed and Annotated Data

Parse the raw data into the three core tables of your addition: the `LIB`, `CORPUS`, and `VOCAB` tables.

These tables will be stored as CSV files with header rows.

You may consider using `|` as a delimitter.

Provide the following information for each.

In [23]:
import re
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction import text
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import PCA
from sklearn.decomposition import LatentDirichletAllocation as LDA
from sklearn.manifold import TSNE as tsne
from gensim.parsing.porter import PorterStemmer
from gensim.models import word2vec

sns.set_theme(style='darkgrid')

# Get the file listing from the GitHub API
api_url = "https://api.github.com/repos/marcomorucci/Clustering-Constitutions/contents/constitutions"
response = requests.get(api_url, headers={"User-Agent": "FinalProject-OHCO"}, timeout=30)
response.raise_for_status()
files = sorted(response.json(), key=lambda x: x["name"])

constitutions = {}
lib_rows = []

for file_info in files:
    if not file_info["name"].endswith(".txt"):
        continue

    source_filename = file_info["name"]
    doc_id = source_filename.removesuffix(".txt")
    match = re.match(r"^(?P<country>.+)_(?P<year>\d{4})$", doc_id)
    country = match.group("country").replace("_", " ") if match else doc_id.replace("_", " ")
    year = int(match.group("year")) if match else np.nan

    raw_url = file_info["download_url"]
    raw_text = requests.get(raw_url, headers={"User-Agent": "FinalProject-OHCO"}, timeout=30).text
    constitutions[doc_id] = raw_text

    clean_lines = [line.strip() for line in raw_text.splitlines() if line.strip() and line.strip() != "Share"]
    lib_rows.append({
        "doc": doc_id,
        "country": country,
        "year": year,
        "source_filename": source_filename,
        "source_url": raw_url,
        "title_line": clean_lines[0] if clean_lines else "",
        "char_len": len(raw_text),
        "line_count": len(clean_lines),
    })

LIB = pd.DataFrame(lib_rows).set_index("doc").sort_index()

print(f"Loaded {len(constitutions)} constitutions")
print(f"Average document length: {LIB['char_len'].mean():,.0f} characters")
LIB.head()


Loaded 192 constitutions
Average document length: 138,099 characters


,country,year,source_filename,source_url,title_line,char_len,line_count
doc,,,,,,,
Afghanistan_2004,Afghanistan,2004,Afghanistan_2004.txt,https://raw.githubusercontent.com/marcomorucci...,Afghanistan 2004,66806,453
Albania_2008,Albania,2008,Albania_2008.txt,https://raw.githubusercontent.com/marcomorucci...,Albania 1998 (rev. 2008),86022,792
Algeria_2008,Algeria,2008,Algeria_2008.txt,https://raw.githubusercontent.com/marcomorucci...,Algeria 1963 (rev. 2008),66590,658
Andorra_1993,Andorra,1993,Andorra_1993.txt,https://raw.githubusercontent.com/marcomorucci...,Andorra 1993,55687,412
Angola_2010,Angola,2010,Angola_2010.txt,https://raw.githubusercontent.com/marcomorucci...,Angola 2010,175148,1393


In [24]:
# Middle OHCO parser: doc -> div1 -> div2 -> unit -> para -> sent -> token
# div1/div2 store flexible high-level legal divisions; unit is Article when present,
# otherwise numbered provisions for article-less constitutions.
HEADING_PATTERNS = {
    "preamble": re.compile(r"^\s*Preamble\b", re.I),
    "title": re.compile(r"^\s*TITLE\s+([IVXLCDM]+|\d+|[A-Z])\b[\.:]?\s*(.*)$", re.I),
    "part": re.compile(r"^\s*PART\s+([IVXLCDM]+|\d+|[A-Z]+)\b[\.:]?\s*(.*)$", re.I),
    "chapter": re.compile(r"^\s*CHAPTER\s+([IVXLCDM]+|\d+|[A-Z]+)\b[\.:]?\s*(.*)$", re.I),
    "section": re.compile(r"^\s*(SECTION|SEC\.?)\s+([IVXLCDM]+|\d+|[A-Z])\b[\.:]?\s*(.*)$", re.I),
    "article": re.compile(r"^\s*(ARTICLE|ART\.?)\s*[\(\[]?([IVXLCDM]+|\d+|[A-Z])\b[\)\]]?[\.:]?\s*(.*)$", re.I),
    "numbered": re.compile(r"^\s*(\d+(?:\.\d+)*)[\).]?\s+(.*)$"),
}
TOKEN_RE = re.compile(r"[A-Za-z]+(?:[-'][A-Za-z]+)*|\d+(?:\.\d+)*")
SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def simple_pos(token_str):
    """Lightweight POS fallback so the CORPUS has pos and pos_group without NLTK data downloads."""
    if token_str.isdigit() or re.fullmatch(r"\d+(?:\.\d+)*", token_str):
        return "CD", "NUM"
    if token_str[:1].isupper():
        return "NNP", "NOUN"
    lower = token_str.lower()
    if lower.endswith("ly"):
        return "RB", "ADV"
    if lower.endswith(("ing", "ed")):
        return "VB", "VERB"
    if lower in {"the", "a", "an"}:
        return "DT", "DET"
    if lower in {"and", "or", "but", "nor"}:
        return "CC", "CONJ"
    if lower in {"in", "of", "to", "for", "with", "by", "from", "on", "at", "under", "over"}:
        return "IN", "ADP"
    return "NN", "NOUN"


def split_sentences(block):
    sentences = [s.strip() for s in SENTENCE_SPLIT_RE.split(block) if s.strip()]
    return sentences or [block.strip()]


def detect_heading(line):
    for heading_type in ["preamble", "title", "part", "chapter", "section", "article"]:
        match = HEADING_PATTERNS[heading_type].match(line)
        if match:
            if heading_type == "preamble":
                return {"type": heading_type, "label": "Preamble", "text": ""}
            if heading_type in {"section", "article"}:
                label = match.group(2)
                trailing = match.group(3).strip()
            else:
                label = match.group(1)
                trailing = match.group(2).strip()
            return {"type": heading_type, "label": label, "text": trailing}
    return None


def add_text_block(rows, doc_id, state, block):
    block = block.strip()
    if not block:
        return

    para_id = state["para"]
    state["para"] += 1

    for sent_id, sentence in enumerate(split_sentences(block)):
        # Keep token ids contiguous after removing leftover source UI artifacts.
        token_strings = [match.group(0) for match in TOKEN_RE.finditer(sentence) if match.group(0) != "Share"]
        for token_id, token_str in enumerate(token_strings):
            pos, pos_group = simple_pos(token_str)
            rows.append({
                "doc": doc_id,
                "div1": state["div1"],
                "div2": state["div2"],
                "unit": state["unit"],
                "para": para_id,
                "sent": sent_id,
                "token": token_id,
                "div1_type": state["div1_type"],
                "div1_label": state["div1_label"],
                "div2_type": state["div2_type"],
                "div2_label": state["div2_label"],
                "unit_type": state["unit_type"],
                "unit_label": state["unit_label"],
                "token_str": token_str,
                "term_str": token_str.lower(),
                "pos": pos,
                "pos_group": pos_group,
            })


def tokenize_ohco(text, doc_id):
    rows = []
    heading_counts = {key: 0 for key in HEADING_PATTERNS}
    raw_lines = [line.strip() for line in text.splitlines() if line.strip()]
    heading_counts["share_line"] = sum(1 for line in raw_lines if line == "Share")
    lines = [line for line in raw_lines if line != "Share"]

    # Prefer Article as the unit when a document uses articles at all; otherwise fall back to numbered provisions.
    has_articles = any(HEADING_PATTERNS["article"].match(line) for line in lines)
    state = {
        "div1": 0, "div2": 0, "unit": 0, "para": 0,
        "div1_type": "document", "div1_label": "whole_document",
        "div2_type": "body", "div2_label": "main_text",
        "unit_type": "intro", "unit_label": "intro",
    }

    for line_id, line in enumerate(lines):
        # First line is usually source metadata like "Afghanistan 2004".
        if line_id == 0 and re.search(r"\b\d{4}\b", line):
            continue

        heading = detect_heading(line)
        if heading:
            heading_counts[heading["type"]] += 1
            htype = heading["type"]

            if htype in {"title", "part"}:
                state["div1"] += 1
                state["div2"] = 0
                state["unit"] = 0
                state["para"] = 0
                state["div1_type"] = htype
                state["div1_label"] = heading["label"]
                state["div2_type"] = "body"
                state["div2_label"] = "main_text"
                state["unit_type"] = "intro"
                state["unit_label"] = "intro"
                add_text_block(rows, doc_id, state, heading["text"])
                continue

            if htype in {"chapter", "section"}:
                # If there is no title/part above it, let chapter be div1. Otherwise chapter/section become div2.
                if htype == "chapter" and state["div1_type"] not in {"title", "part"}:
                    state["div1"] += 1
                    state["div2"] = 0
                    state["div1_type"] = htype
                    state["div1_label"] = heading["label"]
                    state["div2_type"] = "body"
                    state["div2_label"] = "main_text"
                else:
                    state["div2"] += 1
                    state["div2_type"] = htype
                    state["div2_label"] = heading["label"]
                state["unit"] = 0
                state["para"] = 0
                state["unit_type"] = "intro"
                state["unit_label"] = "intro"
                add_text_block(rows, doc_id, state, heading["text"])
                continue

            if htype in {"article", "preamble"}:
                state["unit"] += 1
                state["para"] = 0
                state["unit_type"] = htype
                state["unit_label"] = heading["label"]
                add_text_block(rows, doc_id, state, heading["text"])
                continue

        numbered = HEADING_PATTERNS["numbered"].match(line)
        if numbered and not has_articles:
            heading_counts["numbered"] += 1
            state["unit"] += 1
            state["para"] = 0
            state["unit_type"] = "numbered"
            state["unit_label"] = numbered.group(1)
            add_text_block(rows, doc_id, state, numbered.group(2))
            continue

        add_text_block(rows, doc_id, state, line)

    return rows, heading_counts


ohco_list = []
coverage_rows = []
for doc_id, raw_text in constitutions.items():
    doc_rows, counts = tokenize_ohco(raw_text, doc_id)
    ohco_list.extend(doc_rows)
    coverage_rows.append({"doc": doc_id, **counts})

CORPUS = pd.DataFrame(ohco_list)
CORPUS.set_index(["doc", "div1", "div2", "unit", "para", "sent", "token"], inplace=True)
ohco_df = CORPUS  # Backward-compatible alias for earlier notebook wording.

HEADING_COVERAGE = pd.DataFrame(coverage_rows).set_index("doc").sort_index()


def infer_document_schemes(counts):
    """Summarize each document's structural strategy for LIB."""
    if counts["article"] > 0:
        unit_scheme = "article"
    elif counts["numbered"] > 0:
        unit_scheme = "numbered_provision"
    else:
        unit_scheme = "body_block"

    div1_candidates = [name for name in ["title", "part", "chapter"] if counts[name] > 0]
    div2_candidates = [name for name in ["chapter", "section"] if counts[name] > 0]
    if div1_candidates and div2_candidates and div1_candidates[0] == div2_candidates[0] and len(div2_candidates) > 1:
        div2_candidates = div2_candidates[1:]
    elif div1_candidates and div2_candidates and div1_candidates[0] == div2_candidates[0]:
        div2_candidates = []

    return pd.Series({
        "unit_scheme": unit_scheme,
        "div1_scheme": "+".join(div1_candidates) if div1_candidates else "document",
        "div2_scheme": "+".join(div2_candidates) if div2_candidates else "body",
    })

SCHEME_SUMMARY = HEADING_COVERAGE.apply(infer_document_schemes, axis=1)
LIB = LIB.join(HEADING_COVERAGE.add_prefix("n_")).join(SCHEME_SUMMARY)

print("CORPUS shape:", CORPUS.shape)
print("OHCO index:", "doc -> div1 -> div2 -> unit -> para -> sent -> token")
print("Documents with articles:", int((HEADING_COVERAGE["article"] > 0).sum()))
print("Documents using numbered-provision fallback:", int(((HEADING_COVERAGE["article"] == 0) & (HEADING_COVERAGE["numbered"] > 0)).sum()))
print("Unit schemes:", LIB["unit_scheme"].value_counts().to_dict())
print("Div1 schemes:", LIB["div1_scheme"].value_counts().head(10).to_dict())
print("Div2 schemes:", LIB["div2_scheme"].value_counts().head(10).to_dict())

HEADING_COVERAGE.sum().sort_values(ascending=False), LIB[["country", "year", "div1_scheme", "div2_scheme", "unit_scheme"]].head(), CORPUS.head(20)


CORPUS shape: (4181668, 10)
OHCO index: doc -> div1 -> div2 -> unit -> para -> sent -> token
Documents with articles: 138
Documents using numbered-provision fallback: 53
Unit schemes: {'article': 138, 'numbered_provision': 53, 'body_block': 1}
Div1 schemes: {'part+chapter': 59, 'chapter': 49, 'title+chapter': 31, 'document': 16, 'part': 15, 'title': 13, 'title+part+chapter': 7, 'title+part': 2}
Div2 schemes: {'chapter': 71, 'body': 70, 'chapter+section': 26, 'section': 25}


(share_line    38411
 numbered      35678
 article       22643
 chapter        2291
 section        1893
 part           1467
 title           637
 preamble        161
 dtype: int64,
                       country  year    div1_scheme      div2_scheme  \
 doc                                                                   
 Afghanistan_2004  Afghanistan  2004        chapter             body   
 Albania_2008          Albania  2008   part+chapter          chapter   
 Algeria_2008          Algeria  2008  title+chapter          chapter   
 Andorra_1993          Andorra  1993  title+chapter          chapter   
 Angola_2010            Angola  2010  title+chapter  chapter+section   
 
                  unit_scheme  
 doc                           
 Afghanistan_2004     article  
 Albania_2008         article  
 Algeria_2008         article  
 Andorra_1993         article  
 Angola_2010          article  ,
                                                 div1_type      div1_label  \
 doc    

## LIB (2)

The source documents the corpus comprises. These may be books, plays, newspaper articles, abstracts, blog posts, etc. 

Note that these are *not* documents in the sense used to describe a bag-of-words representation of a text, e.g. chapter.

- UVA Box URL:
- GitHub URL for notebook used to create:
- Delimitter: comma
- Number of observations: 192
- List of features, including at least three that may be used for model summarization (e.g. date, author, etc.): country, year, source_filename, source_url, title_line, char_len, line_count, n_article, n_numbered, n_preamble, div1_scheme, div2_scheme, unit_scheme
- Average length of each document in characters: 138,099

CSV output: [tables/LIB.csv](tables/LIB.csv)


In [25]:
LIB.head()

,country,year,source_filename,source_url,title_line,char_len,line_count,n_preamble,n_title,n_part,n_chapter,n_section,n_article,n_numbered,n_share_line,unit_scheme,div1_scheme,div2_scheme
doc,,,,,,,,,,,,,,,,,,
Afghanistan_2004,Afghanistan,2004,Afghanistan_2004.txt,https://raw.githubusercontent.com/marcomorucci...,Afghanistan 2004,66806,453,1,0,0,12,0,162,0,175,article,chapter,body
Albania_2008,Albania,2008,Albania_2008.txt,https://raw.githubusercontent.com/marcomorucci...,Albania 1998 (rev. 2008),86022,792,1,0,18,12,0,181,0,209,article,part+chapter,chapter
Algeria_2008,Algeria,2008,Algeria_2008.txt,https://raw.githubusercontent.com/marcomorucci...,Algeria 1963 (rev. 2008),66590,658,1,4,0,10,0,182,0,197,article,title+chapter,chapter
Andorra_1993,Andorra,1993,Andorra_1993.txt,https://raw.githubusercontent.com/marcomorucci...,Andorra 1993,55687,412,1,9,0,11,0,107,0,134,article,title+chapter,chapter
Angola_2010,Angola,2010,Angola_2010.txt,https://raw.githubusercontent.com/marcomorucci...,Angola 2010,175148,1393,1,8,0,19,20,244,0,248,article,title+chapter,chapter+section


In [26]:
LIB["char_len"].mean()

np.float64(138099.45833333334)

## CORPUS (2)

The sequence of word tokens in the corpus, indexed by their location in the corpus and document structures.

- UVA Box URL:
- GitHub URL for notebook used to create:
- Delimitter: comma
- Number of observations Between (should be >= 500,000 and <= 2,000,000 observations.): 4,181,668
- OHCO Structure (as delimitted column names): doc|div1|div2|unit|para|sent|token
- Columns (as delimitted column names, including `token_str`, `term_str`, `pos`, and `pos_group`): div1_type|div1_label|div2_type|div2_label|unit_type|unit_label|token_str|term_str|pos|pos_group

CSV output: [tables/CORPUS.csv](tables/CORPUS.csv)


In [27]:
CORPUS.head()

div1_type      div1_label  \
doc              div1 div2 unit para sent token                             
Afghanistan_2004 0    0    1    0    0    0      document  whole_document   
                                          1      document  whole_document   
                                          2      document  whole_document   
                                          3      document  whole_document   
                                          4      document  whole_document   

                                                div2_type div2_label  \
doc              div1 div2 unit para sent token                        
Afghanistan_2004 0    0    1    0    0    0          body  main_text   
                                          1          body  main_text   
                                          2          body  main_text   
                                          3          body  main_text   
                                          4          body  main_text   

                                                unit_type unit_label  \
doc              div1 div2 unit para sent token                        
Afghanistan_2004 0    0    1    0    0    0      preamble   Preamble   
                                          1      preamble   Preamble   
                                          2      preamble   Preamble   
                                          3      preamble   Preamble   
                                          4      preamble   Preamble   

                                                token_str term_str  pos  \
doc              div1 div2 unit para sent token                           
Afghanistan_2004 0    0    1    0    0    0            In       in  NNP   
                                          1           the      the   DT   
                                          2          name     name   NN   
                                          3            of       of   IN   
                                          4         Allah    allah  NNP   

                                                pos_group  
doc              div1 div2 unit para sent token            
Afghanistan_2004 0    0    1    0    0    0          NOUN  
                                          1           DET  
                                          2          NOUN  
                                          3           ADP  
                                          4          NOUN

In [28]:
CORPUS.describe()

,div1_type,div1_label,div2_type,div2_label,unit_type,unit_label,token_str,term_str,pos,pos_group
count,4181668,4181668,4181668,4181668,4181668,4181668,4181668,4181668,4181668,4181668
unique,4,103,3,372,4,520,34199,27096,8,7
top,part,II,body,main_text,article,2,the,the,NN,NOUN
freq,1990012,528788,2608358,2608358,2034963,393441,384898,432314,1915034,2500356


## VOCAB (2)

The unique word types (terms) in the corpus.

- UVA Box URL:
- GitHub URL for notebook used to create:
- Delimitter: comma
- Number of observations: 27,096
- Columns (as delimitted names, including `n`, `p`', `i`, `dfidf`, `porter_stem`, `max_pos` and `max_pos_group`, `stop`): n|p|i|h|df|idf|dfidf|n_chars|ngram_length|porter_stem|max_pos|max_pos_group|n_pos|cat_pos|n_pos_group|cat_pos_group|stop
- Note: Your VOCAB may contain ngrams. If so, add a feature for `ngram_length`.
- List the top 20 significant words in the corpus by DFIDF.

CSV output: [tables/VOCAB.csv](tables/VOCAB.csv)


In [29]:
VOCAB = CORPUS["term_str"].value_counts().to_frame("n").sort_index()
VOCAB.index.name = "term_str"

VOCAB["p"] = VOCAB["n"] / VOCAB["n"].sum()
VOCAB["i"] = -np.log2(VOCAB["p"])
VOCAB["h"] = VOCAB["p"] * VOCAB["i"]
VOCAB["n_chars"] = VOCAB.index.str.len()
VOCAB["ngram_length"] = VOCAB.index.to_series().str.count(r"\s+").add(1).astype("int")

doc_count = CORPUS.index.get_level_values("doc").nunique()
VOCAB["df"] = CORPUS.reset_index().groupby("term_str")["doc"].nunique()
VOCAB["idf"] = np.log2(doc_count / VOCAB["df"])
VOCAB["dfidf"] = VOCAB["df"] * VOCAB["idf"]

pos_counts = CORPUS.groupby(["term_str", "pos"]).size().rename("n_pos_tokens").reset_index()
pos_group_counts = CORPUS.groupby(["term_str", "pos_group"]).size().rename("n_pos_group_tokens").reset_index()

VOCAB["n_pos"] = pos_counts.groupby("term_str")["pos"].nunique()
VOCAB["cat_pos"] = pos_counts.groupby("term_str")["pos"].apply(lambda x: "|".join(sorted(x)))
VOCAB["max_pos"] = pos_counts.sort_values(["term_str", "n_pos_tokens", "pos"], ascending=[True, False, True]) \
    .drop_duplicates("term_str") \
    .set_index("term_str")["pos"]

VOCAB["n_pos_group"] = pos_group_counts.groupby("term_str")["pos_group"].nunique()
VOCAB["cat_pos_group"] = pos_group_counts.groupby("term_str")["pos_group"].apply(lambda x: "|".join(sorted(x)))
VOCAB["max_pos_group"] = pos_group_counts.sort_values(["term_str", "n_pos_group_tokens", "pos_group"], ascending=[True, False, True]) \
    .drop_duplicates("term_str") \
    .set_index("term_str")["pos_group"]

stop_words = set(ENGLISH_STOP_WORDS)
VOCAB["stop"] = VOCAB.index.isin(stop_words).astype("int")

porter = PorterStemmer()
VOCAB["porter_stem"] = [porter.stem(term) for term in VOCAB.index]

VOCAB = VOCAB[[
    "n", "p", "i", "h", "df", "idf", "dfidf", "n_chars", "ngram_length",
    "porter_stem", "max_pos", "max_pos_group", "n_pos", "cat_pos", "n_pos_group", "cat_pos_group", "stop"
]]

print("VOCAB shape:", VOCAB.shape)
print("VOCAB columns:", "|".join(VOCAB.columns))

VOCAB.head()


VOCAB shape: (27096, 17)
VOCAB columns: n|p|i|h|df|idf|dfidf|n_chars|ngram_length|porter_stem|max_pos|max_pos_group|n_pos|cat_pos|n_pos_group|cat_pos_group|stop


,n,p,i,h,df,idf,dfidf,n_chars,ngram_length,porter_stem,max_pos,max_pos_group,n_pos,cat_pos,n_pos_group,cat_pos_group,stop
term_str,,,,,,,,,,,,,,,,,
0,23,5.500198e-06,17.472085,0.000096,8,4.584963,36.679700,1,1,0,CD,NUM,1,CD,1,NUM,0
0.05,1,2.391390e-07,21.995647,0.000005,1,7.584963,7.584963,4,1,0.05,CD,NUM,1,CD,1,NUM,0
0.1,3,7.174171e-07,20.410685,0.000015,2,6.584963,13.169925,3,1,0.1,CD,NUM,1,CD,1,NUM,0
0.15,3,7.174171e-07,20.410685,0.000015,1,7.584963,7.584963,4,1,0.15,CD,NUM,1,CD,1,NUM,0
0.19,1,2.391390e-07,21.995647,0.000005,1,7.584963,7.584963,4,1,0.19,CD,NUM,1,CD,1,NUM,0


In [30]:
VOCAB.sort_values(by='dfidf', ascending=False).head(20).index

Index(['assent', 'occasion', 'accompanied', 'indivisible', 'organized',
       'answer', 'construction', 'holders', 'servants', 'applying', 'holder',
       'm', 'rural', 'stage', 'copy', 'forty-eight', 'resolved', 'reading',
       'delivered', 'hand'],
      dtype='object', name='term_str')

# Derived Tables

## BOW (3)

A bag-of-words representation of the CORPUS.

- UVA Box URL:
- GitHub URL for notebook used to create:
- Delimitter: comma
- Bag (expressed in terms of OHCO levels): selected TFIDF bag DOCS = doc; additional BOW tables include UNITS = doc|div1|div2|unit, PARAS = doc|div1|div2|unit|para, and SENTS = doc|div1|div2|unit|para|sent
- Number of observations: `BOW_UNIT.shape[0]`, `BOW_PARA.shape[0]`, and `BOW_SENT.shape[0]` after execution
- Columns (as delimitted names, including `n`, `tfidf`): `n` is created here; `tf` and `tfidf` are added to `BOW` in the TFIDF section after TF and IDF are computed

CSV output: [tables/BOW.csv](tables/BOW.csv), [tables/BOW_UNIT.csv](tables/BOW_UNIT.csv), [tables/BOW_PARA.csv](tables/BOW_PARA.csv), [tables/BOW_SENT.csv](tables/BOW_SENT.csv)


In [31]:
OHCO = ["doc", "div1", "div2", "unit", "para", "sent", "token"]
bags = dict(
    SENTS = OHCO[:6],
    PARAS = OHCO[:5],
    UNITS = OHCO[:4],
    DOCS = OHCO[:1]
)
bag = 'DOCS'
TOKEN = CORPUS.reset_index()
BOW = TOKEN.groupby(bags[bag]+['term_str']).term_str.count().to_frame('n')
BOW_DOC = BOW
BOW_UNIT = TOKEN.groupby(bags['UNITS']+['term_str']).term_str.count().to_frame('n')
BOW_PARA = TOKEN.groupby(bags['PARAS']+['term_str']).term_str.count().to_frame('n')
BOW_SENT = TOKEN.groupby(bags['SENTS']+['term_str']).term_str.count().to_frame('n')
BOW.head()


n
doc              term_str    
Afghanistan_2004 1         20
                 10         3
                 11         1
                 111        1
                 112        1

In [32]:
BOW_UNIT.head(), BOW_PARA.head(), BOW_SENT.head()


(                                            n
 doc              div1 div2 unit term_str     
 Afghanistan_2004 0    0    1    2004        1
                                 3           1
                                 a           3
                                 accordance  1
                                 adhering    1,
                                                  n
 doc              div1 div2 unit para term_str     
 Afghanistan_2004 0    0    1    0    allah       1
                                      beneficent  1
                                      in          1
                                      merciful    1
                                      most        2,
                                                       n
 doc              div1 div2 unit para sent term_str     
 Afghanistan_2004 0    0    1    0    0    allah       1
                                           beneficent  1
                                           in          1
                    

## DTM (3)

A represenation of the BOW as a sparse count matrix.

- UVA Box URL:
- UVA Box URL of BOW used to generate (if applicable): see above
- GitHub URL for notebook used to create:
- Delimitter: comma
- Bag (expressed in terms of OHCO levels): DOCS = doc

CSV output: [tables/DTCM.csv](tables/DTCM.csv)


In [33]:
DTCM = BOW.n.unstack(fill_value=0)
DTM = DTCM
DOC = DTCM.sum(1).to_frame('n_tokens')
DOC['n_types'] = DTCM.astype('bool').sum(1)
DOC['pkr'] = DOC.n_types / DOC.n_tokens
DOC = DOC.join(LIB[['country', 'year', 'unit_scheme']])
DTCM.head(10)


term_str,0,0.05,0.1,0.15,0.19,0.25,0.3,0.30,0.33,0.35,...,zr,zs,zt,zuba,zug,zui,zulia,zurich,zurmi,zuru
doc,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Albania_2008,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Algeria_2008,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Andorra_1993,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Angola_2010,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Antigua_and_Barbuda_1981,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Argentina_1994,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Armenia_2005,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Australia_1985,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## TFIDF (3)

A Document-Term matrix with TFIDF values.

- UVA Box URL:
- UVA Box URL of DTM or BOW used to create: see above
- GitHub URL for notebook used to create:
- Delimitter: comma
- Description of TFIDIF formula ($\LaTeX$ OK): M05 double-normalized term frequency and standard IDF: `TF = ((DTCM.T + .5) / (DTCM.T.max() + .5)) + .5`, transposed back to document-term form; `IDF = log2(N / DF)` where `N` is the number of bags and `DF` is the number of bags containing the term; `TFIDF = TF * IDF`.

CSV output: [tables/TFIDF.csv](tables/TFIDF.csv)


In [34]:
tf_method = 'double_norm'
tf_norm_k = .5
idf_method = 'standard'
gradient_cmap = 'YlGnBu'

print('TF method:', tf_method)

if tf_method == 'sum':
    TF = DTCM.T / DTCM.T.sum()
elif tf_method == 'smooth':
    TF = (DTCM.T / DTCM.T.sum()) + 1
elif tf_method == 'max':
    TF = DTCM.T / DTCM.T.max()
elif tf_method == 'log':
    TF = np.log2(1 + DTCM.T)
elif tf_method == 'raw':
    TF = DTCM.T
elif tf_method == 'double_norm':
    TF = ((DTCM.T + tf_norm_k) / (DTCM.T.max() + tf_norm_k)) + tf_norm_k
elif tf_method == 'binary':
    TF = DTCM.T.astype('bool').astype('int')

TF = TF.T

DF = DTCM.astype('bool').sum()
N = DTCM.shape[0]

print('IDF method:', idf_method)

if idf_method == 'standard':
    IDF = np.log2(N / DF)
elif idf_method == 'max':
    IDF = np.log2(DF.max() / DF)
elif idf_method == 'plus':
    IDF = np.log2(N / DF) + 1
elif idf_method == 'smooth':
    IDF = np.log2((1 + N) / (1 + DF)) + 1

TFIDF = TF * IDF
BOW['tf'] = TF.stack()
BOW['tfidf'] = TFIDF.stack()
VOCAB['df'] = DF
VOCAB['idf'] = IDF
VOCAB['tfidf_mean'] = TFIDF.mean()
VOCAB['dfidf'] = VOCAB.df * VOCAB.idf
top_20_dfidf = VOCAB.sort_values('dfidf', ascending=False).head(20)
print('BOW bag used for DFIDF:', bag, '=', '|'.join(bags[bag]))
print('Top 20 significant words by BOW-level DFIDF:')
print(', '.join(top_20_dfidf.index.to_list()))
TFIDF.head()


TF method: double_norm
IDF method: standard
BOW bag used for DFIDF: DOCS = doc
Top 20 significant words by BOW-level DFIDF:
assent, occasion, accompanied, indivisible, organized, answer, construction, holders, servants, applying, holder, m, rural, stage, copy, forty-eight, resolved, reading, delivered, hand


term_str,0,0.05,0.1,0.15,0.19,0.25,0.3,0.30,0.33,0.35,...,zr,zs,zt,zuba,zug,zui,zulia,zurich,zurmi,zuru
doc,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,2.294493,3.795809,3.295371,3.795809,3.795809,3.795809,3.295371,3.795809,3.795809,3.795809,...,3.795809,3.795809,3.795809,3.795809,3.795809,3.795809,3.295371,3.295371,3.795809,3.795809
Albania_2008,2.293951,3.794913,3.294592,3.794913,3.794913,3.794913,3.294592,3.794913,3.794913,3.794913,...,3.794913,3.794913,3.794913,3.794913,3.794913,3.794913,3.294592,3.294592,3.794913,3.794913
Algeria_2008,2.293937,3.794890,3.294572,3.794890,3.794890,3.794890,3.294572,3.794890,3.794890,3.794890,...,3.794890,3.794890,3.794890,3.794890,3.794890,3.794890,3.294572,3.294572,3.794890,3.794890
Andorra_1993,2.294529,3.795869,3.295422,3.795869,3.795869,3.795869,3.295422,3.795869,3.795869,3.795869,...,3.795869,3.795869,3.795869,3.795869,3.795869,3.795869,3.295422,3.295422,3.795869,3.795869
Angola_2010,2.293284,3.793809,3.293634,3.793809,3.793809,3.793809,3.293634,3.793809,3.793809,3.793809,...,3.793809,3.793809,3.793809,3.793809,3.793809,3.793809,3.293634,3.293634,3.793809,3.793809


## Reduced and Normalized TFIDF_L2 (3)

CSV output: [tables/TFIDF_L2.csv](tables/TFIDF_L2.csv)


In [35]:
VOCAB['dfidf'] = VOCAB.df * VOCAB.idf
VOCAB['dp'] = VOCAB.df / N
VOCAB['di'] = np.log2(1/VOCAB.dp)
VOCAB['dh'] = VOCAB.dp * VOCAB.di

thresh = VOCAB.dh.quantile(.9).round(3)
SIGS = VOCAB[(VOCAB.dh >= thresh) & (VOCAB.stop == 0)]
TFIDF_REDUCED = TFIDF[SIGS.index]
TFIDF_L2 = TFIDF_REDUCED.T / np.sqrt((TFIDF_REDUCED ** 2).sum(1))
TFIDF_L2 = TFIDF_L2.T.fillna(0)
TFIDF_L2


term_str,000,100,101,102,103,104,105,106,107,108,...,works,world,worship,writ,writs,x,young,youth,zone,zones
doc,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,0.021680,0.019541,0.022696,0.025365,0.022349,0.023783,0.023412,0.025787,0.024554,0.022696,...,0.013876,0.011844,0.011672,0.025787,0.022349,0.027127,0.018708,0.016924,0.021680,0.028090
Albania_2008,0.021737,0.019542,0.022697,0.025367,0.022351,0.023816,0.023474,0.025789,0.024556,0.022697,...,0.013877,0.011845,0.011673,0.025789,0.022351,0.027129,0.018757,0.016925,0.021765,0.028164
Algeria_2008,0.021682,0.019543,0.022698,0.025368,0.022352,0.023786,0.023415,0.025790,0.024557,0.022698,...,0.013878,0.011890,0.011673,0.025790,0.022352,0.027130,0.018710,0.016947,0.021737,0.028201
Andorra_1993,0.021681,0.019543,0.022698,0.025367,0.022391,0.023786,0.023414,0.025881,0.024556,0.022698,...,0.013902,0.011866,0.011673,0.025789,0.022351,0.027130,0.018710,0.016925,0.021681,0.028093
Angola_2010,0.021693,0.019539,0.022693,0.025362,0.022347,0.023781,0.023410,0.025784,0.024551,0.022693,...,0.013894,0.011843,0.011695,0.025838,0.022347,0.027125,0.018772,0.017005,0.021768,0.028127
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Vanuatu_1983,0.021677,0.019539,0.022693,0.025362,0.022347,0.023781,0.023410,0.025784,0.024551,0.022693,...,0.013875,0.011843,0.011702,0.025784,0.022407,0.027124,0.018756,0.016922,0.021677,0.028087
Venezuela_2009,0.021674,0.019536,0.022690,0.025358,0.022343,0.023777,0.023419,0.025780,0.024547,0.022690,...,0.013928,0.011854,0.011669,0.025780,0.022343,0.027120,0.018714,0.016919,0.021723,0.028083
Yemen_2001,0.021679,0.019540,0.022731,0.025364,0.022349,0.023783,0.023449,0.025786,0.024592,0.022731,...,0.013876,0.011844,0.011690,0.025786,0.022349,0.027126,0.018767,0.016950,0.021714,0.028089


A Document-Term matrix with L2 normalized TFIDF values.

- UVA Box URL:
- UVA Box URL of source TFIDF table: see above
- GitHub URL for notebook used to create:
- Delimitter: comma
- Number of features (i.e. significant words): 2616
- Principle of significant word selection: non-stopword terms with document entropy `dh` at or above the 90th percentile, using the aggregate TFIDF method
